# Reconciliation: Postgres vs OpenSearch

Answers: which papers exist in Postgres (with parsed `raw_text`) but are
MISSING from the OpenSearch index? Run this any time retrieval seems to be
missing content you expect to be there — cheaper than guessing.

**Prereqs:** `docker compose up -d postgres opensearch`

In [1]:
from dotenv import load_dotenv
load_dotenv()

from src.models.paper import Paper
from src.db.factory import make_database
from src.services.opensearch.factory import make_opensearch_client

db = make_database()
os_client = make_opensearch_client()

with db.get_session() as session:
    all_papers   = {p.arxiv_id: p for p in session.query(Paper).all()}
    parsed_ids   = {p.arxiv_id for p in all_papers.values() if p.raw_text}
    unparsed_ids = set(all_papers) - parsed_ids

agg = os_client.client.search(
    index=os_client.index_name,
    body={"size": 0, "aggs": {"papers": {"terms": {"field": "arxiv_id", "size": 1000}}}},
)
indexed = {b["key"]: b["doc_count"] for b in agg["aggregations"]["papers"]["buckets"]}

missing_from_index = parsed_ids - set(indexed)

print(f"Total papers in Postgres      : {len(all_papers)}")
print(f"  parsed (have raw_text)      : {len(parsed_ids)}")
print(f"  NOT parsed (raw_text=NULL)  : {len(unparsed_ids)}")
print(f"Total papers in OpenSearch    : {len(indexed)}")
print(f"Parsed but MISSING from index : {len(missing_from_index)}")
print()

print(f"{'arxiv_id':16} {'in PG':6} {'raw_text':9} {'in OS':6} {'chunks':7} title")
for aid in sorted(all_papers):
    p = all_papers[aid]
    has_text = "yes" if aid in parsed_ids else "NULL"
    in_os    = "yes" if aid in indexed else "NO"
    chunks   = indexed.get(aid, 0)
    flag     = "  <-- needs re-index" if aid in missing_from_index else (
               "  <-- needs re-parse" if aid in unparsed_ids else "")
    print(f"{aid:16} {'yes':6} {has_text:9} {in_os:6} {chunks:<7}{(p.title or '')[:40]}{flag}")

Total papers in Postgres      : 4
  parsed (have raw_text)      : 3
  NOT parsed (raw_text=NULL)  : 1
Total papers in OpenSearch    : 3
Parsed but MISSING from index : 0

arxiv_id         in PG  raw_text  in OS  chunks  title
2605.28807v1     yes    NULL      NO     0      Calibrating Conservatism for Scalable Ov  <-- needs re-parse
2605.28812v1     yes    yes       yes    22     Beyond Binary: Sim-to-Real Dexterous Man
2606.07515v1     yes    yes       yes    14     How reliable are LLMs when it comes to p
2607.08763v1     yes    yes       yes    24     OpenCoF: Learning to Reason Through Vide


In [ ]:
# Auto-fix: re-index anything parsed-but-missing (skips papers with raw_text=NULL)
import asyncio
from src.services.indexing.factory import make_hybrid_indexing_service

if missing_from_index:
    papers_to_fix = [
        {
            "id": str(all_papers[aid].id), "arxiv_id": aid, "title": all_papers[aid].title,
            "authors": all_papers[aid].authors, "abstract": all_papers[aid].abstract,
            "categories": all_papers[aid].categories, "published_date": all_papers[aid].published_date,
            "raw_text": all_papers[aid].raw_text, "sections": all_papers[aid].sections,
        }
        for aid in missing_from_index
    ]
    indexer = make_hybrid_indexing_service()
    stats = await indexer.index_papers_batch(papers=papers_to_fix, replace_existing=True)
    print(stats)
else:
    print("Nothing to fix — every parsed paper is indexed.")